In [82]:
import pandas as pd
df=pd.read_csv('../data/batch2_contracts_125rows.csv')
df.head()

,contract_id,raw_text,apr,term_months,monthly_payment,penalty_clause,recommended_action,risk_flag
0,1,This contract between Prime Auto and Emily Sto...,10.49,24,959,NaN,Approve,Low
1,2,This contract between ABC Motors and Priya Sha...,3.78,24,804,Early termination fee $300,Approve,Low
2,3,This contract between DriveEasy Finance and Sa...,5.41,24,774,Late fee $50,Reject,High
3,4,This contract between National Motors and Alex...,5.26,48,1028,Late fee $25,Approve,High
4,5,This contract between National Motors and John...,5.97,36,1180,NaN,Approve,Low


In [83]:
df["expected_apr"] = df["apr"]
df["expected_term"] = df["term_months"]
df["expected_payment"] = df["monthly_payment"]
df["expected_penalty"] = df["penalty_clause"]

In [84]:
df[['penalty_clause',"expected_penalty"]]

,penalty_clause,expected_penalty
0,NaN,NaN
1,Early termination fee $300,Early termination fee $300
2,Late fee $50,Late fee $50
3,Late fee $25,Late fee $25
4,NaN,NaN
...,...,...
120,Late fee $25,Late fee $25
121,Late fee $25,Late fee $25
122,Early termination fee $300,Early termination fee $300
123,NaN,NaN


In [85]:
df["apr_score"] = (df["apr"]==df["expected_apr"]).astype(int)

df["term_score"] = (df["term_months"]==df["expected_term"]).astype(int)

df["payment_score"] = (df["monthly_payment"]==df["expected_payment"]).astype(int)

df["penalty_score"] = (df["penalty_clause"] == df["expected_penalty"]).astype(int)

df["total_score"] = (df["apr_score"] + df["term_score"] + df["payment_score"] + df["penalty_score"])


In [86]:
df["quality_score"] = (df["total_score"]/3) * 100


In [87]:
df[["apr_score" ,"term_score","payment_score","penalty_score","total_score","quality_score"]].head(20)

,apr_score,term_score,payment_score,penalty_score,total_score,quality_score
0,1,1,1,0,3,100.000000
1,1,1,1,1,4,133.333333
2,1,1,1,1,4,133.333333
3,1,1,1,1,4,133.333333
4,1,1,1,0,3,100.000000
5,1,1,1,0,3,100.000000
6,1,1,1,0,3,100.000000
7,1,1,1,1,4,133.333333
8,1,1,1,0,3,100.000000
9,1,1,1,1,4,133.333333


In [88]:
import requests

In [89]:
def get_vehicle_details(vin):
    pass

In [90]:
import requests

def get_vehicle_details(vin):
    url = f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValues/{vin}?format=json"
    response = requests.get(url)
    data = response.json()
    return data["Results"][0]

In [91]:
vehicle = get_vehicle_details("1HGCM82633A004352")
vehicle

{'ABS': '',
 'ActiveSafetySysNote': '',
 'AdaptiveCruiseControl': '',
 'AdaptiveDrivingBeam': '',
 'AdaptiveHeadlights': '',
 'AdditionalErrorText': '',
 'AirBagLocCurtain': '1st and 2nd Rows',
 'AirBagLocFront': '1st Row (Driver and Passenger)',
 'AirBagLocKnee': '',
 'AirBagLocSeatCushion': '',
 'AirBagLocSide': '1st Row (Driver and Passenger)',
 'AutoReverseSystem': '',
 'AutomaticPedestrianAlertingSound': '',
 'AxleConfiguration': '',
 'Axles': '',
 'BasePrice': '',
 'BatteryA': '',
 'BatteryA_to': '',
 'BatteryCells': '',
 'BatteryInfo': '',
 'BatteryKWh': '',
 'BatteryKWh_to': '',
 'BatteryModules': '',
 'BatteryPacks': '',
 'BatteryType': '',
 'BatteryV': '',
 'BatteryV_to': '',
 'BedLengthIN': '',
 'BedType': 'Not Applicable',
 'BlindSpotIntervention': '',
 'BlindSpotMon': '',
 'BodyCabType': 'Not Applicable',
 'BodyClass': 'Coupe',
 'BrakeSystemDesc': '',
 'BrakeSystemType': '',
 'BusFloorConfigType': 'Not Applicable',
 'BusLength': '',
 'BusType': 'Not Applicable',
 'CAN_AACN

In [92]:
important_vehicle_info = {
    "make": vehicle.get("Make"),
    "model": vehicle.get("Model"),
    "year": vehicle.get("ModelYear"),
    "body_type": vehicle.get("BodyClass")
    }

important_vehicle_info

{'make': 'HONDA', 'model': 'Accord', 'year': '2003', 'body_type': 'Coupe'}

In [93]:
vehicle_year = int(important_vehicle_info["year"])

if vehicle_year < 2015:
    risk_level = "HIGH"

else:
    risk_level = "NORMAL"

risk_level

'HIGH'

In [94]:
clean_vehicle_data = {
    "make": vehicle.get("Make"),
    "model": vehicle.get("Model"),
    "year": vehicle.get("ModelYear"),
    "body_type": vehicle.get("BodyClass")
    }

clean_vehicle_data

{'make': 'HONDA', 'model': 'Accord', 'year': '2003', 'body_type': 'Coupe'}

In [95]:
import pandas as pd 

contracts_df = pd.read_csv("../data/sample_car_contracts.csv")
contracts_df.head()

,id,customer_name,contract_type,vehicle_type,monthly_emi,interest_rate,tenure_months,clause_summary,risk_flag,issue_type,recommended_action
0,1,Rahul Mehta,Car Loan,Sedan,18500,9.5,48,Prepayment allowed only after 24 months with 5...,medium,High prepayment charges,Highlight prepayment penalty to user and sugge...
1,2,Anita Rao,Car Lease,SUV,22000,0.0,36,Lessee must pay for all maintenance and insurance,low,Standard maintenance clause,"No action, just explain maintenance responsibi..."
2,3,James Wilson,Car Loan,Hatchback,14500,11.2,60,Late payment fee of 3% per month on outstandin...,high,Aggressive late fee,Flag clause and suggest user request cap on la...
3,4,Meena Iyer,Car Lease,Sedan,21000,0.0,24,"Excess mileage charge of ₹12 per km over 15,00...",medium,High excess mileage rate,Warn user about extra mileage charges and reco...
4,5,Arjun Patel,Car Loan,SUV,27500,10.8,72,Floating interest rate linked to lender's inte...,high,Unclear interest benchmark,Explain floating rate risk and suggest asking ...


In [96]:

def enrich_contract_with_vehicle(contract_row):
    vin = contract_row["vin"]
    vehicle = get_vehicle_details(vin)

    return {
        "contract_id": contract_row["contract_id"],
        "customer_name": contract_row["customer_name"],
        "vin": vin,
        "vehicle": {
            "make": vehicle.get("Make"),
            "model": vehicle.get("Model"),
            "year": vehicle.get("ModelYear")
        }
    }

In [97]:
def extract_vehicle_info(vehicle_raw):
    """Return a small dict with the vehicle fields we care about."""
    if not vehicle_raw:
        return {"make": None, "model": None, "year": None}
    return {
        "make": vehicle_raw.get("Make"),
        "model": vehicle_raw.get("Model"),
        "year": vehicle_raw.get("ModelYear")
    }


def extract_sla_details(row):
    """Return a compact SLA dict using available contract columns."""
    return {
        "contract_type": row.get("contract_type"),
        "tenure_months": row.get("tenure_months"),
        "monthly_emi": row.get("monthly_emi"),
        "interest_rate": row.get("interest_rate"),
        "vehicle_type": row.get("vehicle_type")
    }


def enrich_contract_with_vehicle(row):
    # Note: sample_car_contracts.csv doesn't have VIN column, so vehicle will be None
    vin = row.get("vin")
    vehicle_raw = get_vehicle_details(vin) if pd.notna(vin) else {}
    vehicle_info = extract_vehicle_info(vehicle_raw)

    return {
        "contract_id": row.get("id"),
        "customer_name": row.get("customer_name"),
        "contract_type": row.get("contract_type"),
        "vehicle_type": row.get("vehicle_type"),
        "sla": extract_sla_details(row),
        "vehicle": vehicle_info,
        "risk_flag": row.get("risk_flag"),
        "issue_type": row.get("issue_type"),
        "recommended_action": row.get("recommended_action")
    }


combined_records = [enrich_contract_with_vehicle(row) for _, row in contracts_df.iterrows()]
combined_df = pd.DataFrame(combined_records)
combined_df.head()

,contract_id,customer_name,contract_type,vehicle_type,sla,vehicle,risk_flag,issue_type,recommended_action
0,1,Rahul Mehta,Car Loan,Sedan,"{'contract_type': 'Car Loan', 'tenure_months':...","{'make': None, 'model': None, 'year': None}",medium,High prepayment charges,Highlight prepayment penalty to user and sugge...
1,2,Anita Rao,Car Lease,SUV,"{'contract_type': 'Car Lease', 'tenure_months'...","{'make': None, 'model': None, 'year': None}",low,Standard maintenance clause,"No action, just explain maintenance responsibi..."
2,3,James Wilson,Car Loan,Hatchback,"{'contract_type': 'Car Loan', 'tenure_months':...","{'make': None, 'model': None, 'year': None}",high,Aggressive late fee,Flag clause and suggest user request cap on la...
3,4,Meena Iyer,Car Lease,Sedan,"{'contract_type': 'Car Lease', 'tenure_months'...","{'make': None, 'model': None, 'year': None}",medium,High excess mileage rate,Warn user about extra mileage charges and reco...
4,5,Arjun Patel,Car Loan,SUV,"{'contract_type': 'Car Loan', 'tenure_months':...","{'make': None, 'model': None, 'year': None}",high,Unclear interest benchmark,Explain floating rate risk and suggest asking ...


In [98]:
combined_df.to_csv("../data/milestone2_evaluation_output.csv", index=False)